In [2]:
from Bio.Seq import Seq
from Bio import SeqIO
import pandas as pd
import sys
import numpy as np

In [ ]:
def find_orfs(fasta: str, user_table : int=1) -> pd.DataFrame:
    """
    Find the longest open reading frames (ORFs) for sequences in a FASTA file

    Args:
        fasta (str): The FASTA file entered by the user on the command line
    Returns:
        pd.DataFrame: A Pandas DataFrame that contains the longest ORF of each sequence, where each ORF represented as:
            - 'sequence' (str): The open reading frame nucleotide sequence.
            - 'reverse' (bool): If it's a forward or reverse sequence
            - 'start' (int): Start position
            - 'end' (int): End position
            - 'frame' (int): Reading frame (1, 2, 3)
            - 'orf' (str): The longest open reading frame
    """
    """ List of Longest Open Reading Frame for FASTA Sequences"""
    orf_list = []

    try:
        with open(fasta, "r") as fh:
            orf_list = [find_longest_orf(record.seq, record.id, user_table).head(1) for record in SeqIO.parse(fh, "fasta")]
    except FileNotFoundError:
        sys.stderr.write("FASTA file was not found")
        raise FileNotFoundError
    except IOError as e:
        sys.stderr.write(f"An IO error has occurred {e}")
        raise IOError
    except Exception as e:
        sys.stderr.write(f"An error has occurred: {e}")
        raise Exception
    return pd.concat(orf_list, ignore_index=True)

def find_longest_orf(seq: str, id : str, user_table : int=1) -> pd.Series:
    """
    Find the longest open reading frames (ORF) in a nucleotide sequence on a given strand.

    Args:
        seq (str): The nucleotide sequence
        fasta_dict (dict): The dictionary that contains the longest ORFs from FASTA file
        reverse (bool): If the nucleotide sequence is forward or reverse
    
    Returns:
        pd.DataFrame : Pandas DataFrame containing ORFs where the longest ORF is at top
    """
    
    stop_codon = ["TAG", "TAA", "TGA"]

    """ FASTA dictonary """
    fasta_dict = {
        "id" : [],
        "start" : [],
        "end" : [],
        "seq" : [],
        "orf" : [],
        "frame" : [],
        "reverse" : [],
        "Amino_Acids" : []
    }

    """ Forward and Reverse Strand """
    for nucleotide in [seq ,seq.reverse_complement()]:
        """ Reading frames 0, 1, 2 """
        for frame in range(3):
            """ Go through whole sequence """
            i = frame
            for i in range(frame, len(nucleotide) - 2, 3):
                """ If a start codon is found - go through rest of sequence for stop codon """
                if nucleotide[i:i + 3] == "ATG":
                    # If another start codon is found in an existing ORF, skip it
                    for j in range(i + 3, len(nucleotide) - 2, 3):
                        """ Add reading frame into FASTA dictionary if stop codon is found """
                        if nucleotide[j:j + 3] in stop_codon:
                            fasta_dict["id"].append(id)
                            fasta_dict["start"].append(i)
                            fasta_dict["end"].append(j + 3)
                            fasta_dict["seq"].append(nucleotide)
                            fasta_dict["orf"].append(nucleotide[i: j + 3])
                            fasta_dict["frame"].append(frame)
                            fasta_dict["Amino_Acids"].append(Seq(nucleotide[i: j + 3]).translate(table=user_table))
                            break # Found ORF, break out of for loop
    print(fasta_dict)
    print(len(fasta_dict["start"]))
    print(len(fasta_dict["end"]))
    print(len(fasta_dict["id"]))
    print(len(fasta_dict["orf"]))
    print(len(fasta_dict["frame"]))
    print(len(fasta_dict["Amino_Acids"]))
    """ Sort Dictionary to where longest ORF is at top """
    return pd.DataFrame(fasta_dict)

In [17]:
test_fasta = "test.fasta"
df1 = find_orfs(test_fasta)
df1

{'id': ['Seq1_long_ALL6_frames', 'Seq1_long_ALL6_frames', 'Seq1_long_ALL6_frames', 'Seq1_long_ALL6_frames', 'Seq1_long_ALL6_frames', 'Seq1_long_ALL6_frames', 'Seq1_long_ALL6_frames', 'Seq1_long_ALL6_frames', 'Seq1_long_ALL6_frames', 'Seq1_long_ALL6_frames', 'Seq1_long_ALL6_frames', 'Seq1_long_ALL6_frames', 'Seq1_long_ALL6_frames', 'Seq1_long_ALL6_frames', 'Seq1_long_ALL6_frames', 'Seq1_long_ALL6_frames', 'Seq1_long_ALL6_frames', 'Seq1_long_ALL6_frames', 'Seq1_long_ALL6_frames', 'Seq1_long_ALL6_frames', 'Seq1_long_ALL6_frames', 'Seq1_long_ALL6_frames', 'Seq1_long_ALL6_frames', 'Seq1_long_ALL6_frames', 'Seq1_long_ALL6_frames', 'Seq1_long_ALL6_frames', 'Seq1_long_ALL6_frames', 'Seq1_long_ALL6_frames'], 'start': [0, 18, 51, 84, 165, 231, 64, 91, 103, 130, 148, 205, 214, 223, 244, 41, 122, 185, 188, 260, 272, 281, 51, 90, 147, 192, 64, 130], 'end': [12, 33, 57, 186, 186, 240, 73, 100, 115, 145, 157, 256, 256, 256, 256, 62, 134, 203, 203, 269, 290, 290, 102, 102, 171, 201, 88, 268], 'seq': [

An error has occurred: All arrays must be of the same length

Exception: 

In [8]:
import pandas as pd

def build_binary_tree_from_order(order: list):
    """
    Build a progressive binary tree from ordering.
    Returns:
        terminals : {1: label, 2: label, ...}
        internals : {nodeId: [leftChild, rightChild]}
    """
    terminals = {i+1: name for i, name in enumerate(order)}
    n = len(order)

    if n <= 1:
        return terminals, {}

    internal_nodes = {}
    next_id = n + 1

    # First join
    internal_nodes[next_id] = [order[0], order[1]]
    prev = next_id
    next_id += 1

    # Add remaining sequences
    for i in range(2, n):
        internal_nodes[next_id] = [prev, order[i]]
        prev = next_id
        next_id += 1

    return terminals, internal_nodes



def get_kmers(seq: str, k: int = 4) -> list:
    """Return list of k-mers from a sequence."""
    return [seq[i:i+k] for i in range(len(seq) - k + 1)]



def jaccard_calculation(kmers1: list, kmers2: list) -> float:
    """
    Calculate Jaccard similarity between two k-mer lists.
    """
    if not kmers1 or not kmers2:
        return 0.0

    set1 = set(kmers1)
    set2 = set(kmers2)

    intersection = len(set1 & set2)
    union = len(set1 | set2)

    if union == 0:
        return 0.0

    return intersection / union



def dataframe_to_dict(df: pd.DataFrame, k=4) -> dict:
    """
    Convert dataframe containing id + Amino_Acids into
    { "id" : [kmers...] }
    """
    kmer_dict = {}
    for idx, row in df.iterrows():
        aa_seq = str(row["Amino_Acids"])
        kmer_dict[str(row["id"])] = get_kmers(aa_seq, k)
    return kmer_dict



def order_by_kmer_similarity(kmer_dict: dict) -> list:
    """
    Order sequences by progressive nearest-neighbor based on k-mer Jaccard similarity.
    """
    names = list(kmer_dict.keys())
    n = len(names)

    if n <= 1:
        return names

    # Precompute pairwise similarities
    sim = {}
    for i in range(n):
        for j in range(i+1, n):
            a, b = names[i], names[j]
            sim[(a, b)] = jaccard_calculation(kmer_dict[a], kmer_dict[b])

    # 1) Find most similar pair
    best_pair = None
    best_val = -1.0
    for pair, v in sim.items():
        if v > best_val:
            best_val = v
            best_pair = pair

    ordered = [best_pair[0], best_pair[1]]
    remaining = set(names) - set(ordered)

    # 2) Add remaining sequences by maximum similarity to the cluster
    while remaining:
        best_option = None
        best_score = -1.0

        for cand in remaining:
            score = max(
                sim.get((a, cand), sim.get((cand, a), 0.0))
                for a in ordered
            )
            if score > best_score:
                best_score = score
                best_option = cand

        ordered.append(best_option)
        remaining.remove(best_option)

    return ordered



# ----- RUN PIPELINE -----
kmer_dict = dataframe_to_dict(df1, k=20)
print(kmer_dict)

ordered = order_by_kmer_similarity(kmer_dict)
print("Order:", ordered)

terminals, internal_nodes = build_binary_tree_from_order(ordered)
print("Terminals:", terminals)
print("Internal Nodes:", internal_nodes)


{'Seq1_long_ALL6_frames': ['MVNAIPWFKLRSSKHGSVTV', 'VNAIPWFKLRSSKHGSVTVS', 'NAIPWFKLRSSKHGSVTVSC', 'AIPWFKLRSSKHGSVTVSCP', 'IPWFKLRSSKHGSVTVSCPN', 'PWFKLRSSKHGSVTVSCPNS', 'WFKLRSSKHGSVTVSCPNSS', 'FKLRSSKHGSVTVSCPNSSG', 'KLRSSKHGSVTVSCPNSSGI', 'LRSSKHGSVTVSCPNSSGIS', 'RSSKHGSVTVSCPNSSGISY', 'SSKHGSVTVSCPNSSGISYP', 'SKHGSVTVSCPNSSGISYPI', 'KHGSVTVSCPNSSGISYPIR', 'HGSVTVSCPNSSGISYPIRP', 'GSVTVSCPNSSGISYPIRPW', 'SVTVSCPNSSGISYPIRPWS', 'VTVSCPNSSGISYPIRPWSS', 'TVSCPNSSGISYPIRPWSSY', 'VSCPNSSGISYPIRPWSSYI', 'SCPNSSGISYPIRPWSSYIQ', 'CPNSSGISYPIRPWSSYIQP', 'PNSSGISYPIRPWSSYIQPH', 'NSSGISYPIRPWSSYIQPHG', 'SSGISYPIRPWSSYIQPHGT', 'SGISYPIRPWSSYIQPHGTL', 'GISYPIRPWSSYIQPHGTL*'], 'Seq2_long_ALL6_frames': ['MVMAIYLSIPWFKHLSAWSK', 'VMAIYLSIPWFKHLSAWSKH', 'MAIYLSIPWFKHLSAWSKHP', 'AIYLSIPWFKHLSAWSKHP*'], 'Seq3_long_ALL6_frames': ['MKLRPWLRMTLDLCVKDDHD', 'KLRPWLRMTLDLCVKDDHDL', 'LRPWLRMTLDLCVKDDHDLD', 'RPWLRMTLDLCVKDDHDLDL', 'PWLRMTLDLCVKDDHDLDLC', 'WLRMTLDLCVKDDHDLDLCG', 'LRMTLDLCVKDDHDLDLCGK', 'RMTLDL

In [14]:
import numpy as np

def align_two(seqA, seqB):
    """
    Align two sequences (both should be numpy arrays of chars).
    Returns np.array([alignedA, alignedB]).
    """
    mat = init_mat(seqA, seqB)
    mat = fill_matrix(mat, seqA, seqB)
    aligned = trace_matrix(mat, seqA, seqB)
    return aligned


def merge_alignment(child1_align, child2_align):
    """
    Merge two aligned children into a single aligned sequence for the parent.
    We only keep the FIRST aligned sequence (typical for progressive alignment).
    """
    # child alignment shape: (2, alignment_length)
    # We need a CONSENSUS or choose one sequence: here just child1
    return child1_align[0]


def progressive_align(df, terminals, internal_nodes):
    """
    Perform progressive alignment following the binary tree.
    """
    # 1. Load leaf sequences as numpy arrays
    seq_dict = parse_fa(df, terminals)

    # 2. Storage for aligned sequences (both terminals & internals)
    aligned = {name: seq_dict[name] for name in seq_dict}

    # 3. Sort internal nodes by increasing ID (bottom-up)
    for node_id in sorted(internal_nodes.keys()):
        left, right = internal_nodes[node_id]

        # LEFT child can be terminal label or internal node numeric ID
        seqL = aligned[left] if isinstance(left, str) else aligned[left]

        # RIGHT child
        seqR = aligned[right] if isinstance(right, str) else aligned[right]

        # 4. Perform pairwise alignment
        aln = align_two(seqL, seqR)

        # 5. Add consensus sequence to dictionary
        aligned[node_id] = merge_alignment(aln[0], aln[1])

    return aligned


In [21]:

def parse_fa(df: pd.DataFrame, use_amino_acids=True):
    """
    Convert ORF DataFrame into:
        { "seqID_orf#", numpy_array_of_characters }

    Parameters
    ----------
    df : pd.DataFrame
        Must contain at least:
            - 'id'           (FASTA sequence id)
            - 'Amino_Acids'
            - 'orf'         (if nucleotide alignment needed)
    use_amino_acids : bool
        If True  → align amino acids
        If False → align nucleotide ORFs

    Returns
    -------
    dict:
        Keys are labels like "seq1_orf0"
        Values are NumPy arrays of characters
    """

    seq_dict = {}

    for idx, row in df.iterrows():
        seq_dict[str(row["id"])] = np.array(list(row["Amino_Acids"]))

    return seq_dict

def init_mat(seq1: np.ndarray, seq2: np.ndarray) -> np.ndarray:
	rows = len(seq2) + 1
	cols = len(seq1) + 1
	return np.zeros((rows, cols), dtype=int)
	

def fill_matrix(matrix, seq1, seq2, match=1, mismatch=-1, indel=-1):
    rows, cols = matrix.shape

    # initialize boundaries
    for i in range(1, rows):
        matrix[i, 0] = matrix[i-1, 0] + indel
    for j in range(1, cols):
        matrix[0, j] = matrix[0, j-1] + indel

    # fill matrix
    for i in range(1, rows):
        for j in range(1, cols):

            if seq1[j-1] == seq2[i-1]:
                diag = matrix[i-1, j-1] + match
            else:
                diag = matrix[i-1, j-1] + mismatch

            up = matrix[i-1, j] + indel
            left = matrix[i, j-1] + indel

            matrix[i, j] = max(diag, up, left)

    return matrix


def trace_matrix(matrix, seq1, seq2):
    i = matrix.shape[0] - 1
    j = matrix.shape[1] - 1

    aligned1 = []
    aligned2 = []

    while i > 0 or j > 0:

        # boundary cases
        if i == 0:
            aligned1.append(seq1[j-1])
            aligned2.append('-')
            j -= 1
            continue

        if j == 0:
            aligned1.append('-')
            aligned2.append(seq2[i-1])
            i -= 1
            continue

        current = matrix[i, j]
        diag = matrix[i-1, j-1]
        up   = matrix[i-1, j]
        left = matrix[i, j-1]

        # match/mismatch score
        score_diag = diag + (1 if seq1[j-1] == seq2[i-1] else -1)

        if current == score_diag:
            aligned1.append(seq1[j-1])
            aligned2.append(seq2[i-1])
            i -= 1
            j -= 1
        elif current == left - 1:
            aligned1.append(seq1[j-1])
            aligned2.append('-')
            j -= 1
        else:
            aligned1.append('-')
            aligned2.append(seq2[i-1])
            i -= 1

    return np.array([aligned1[::-1], aligned2[::-1]])


In [18]:
def progressive_align(order, seq_dict):
    """
    Align sequences following the progressive guide tree order.
    Returns final multiple alignment (dict).
    """
    # Start with first two
    s1 = seq_dict[order[0]]
    s2 = seq_dict[order[1]]

    M = init_mat(s1, s2)
    F = fill_matrix(M, s1, s2)
    align = trace_matrix(F, s1, s2)

    # store aligned sequences
    aligned = {
        order[0]: align[0],
        order[1]: align[1]
    }

    # progressively add others
    for name in order[2:]:
        new = seq_dict[name]

        # build profile → new alignment
        prof_len = len(aligned[order[0]])
        # build consensus-profile representation
        prof_string = np.array([c[0] for c in zip(*aligned.values())])

        M = init_mat(prof_string, new)
        F = fill_matrix(M, prof_string, new)
        result = trace_matrix(F, prof_string, new)

        # add gaps to all sequences to match new alignment
        gap_mask = result[0] == '-'

        # apply gap mask to all existing profiles
        for key in aligned:
            aligned[key] = np.insert(aligned[key], np.where(gap_mask)[0], '-')

        # add new sequence
        aligned[name] = result[1]

    return aligned


In [23]:
    # convert ORFs → sequence arrays
seq_dict = parse_fa(df1, use_amino_acids=True)


    # final progressive multiple alignment
alignment = progressive_align(ordered, seq_dict)
print(alignment)

{'Seq1_long_ALL6_frames': array(['M', 'V', 'N', 'A', 'I', 'P', 'W', 'F', 'K', 'L', 'R', 'S', 'S',
       'K', 'H', 'G', 'S', 'V', 'T', '-', 'V', 'S', 'C', 'P', 'N', 'S',
       'S', 'G', 'I', 'S', 'Y', 'P', 'I', 'R', 'P', 'W', 'S', 'S', 'Y',
       'I', 'Q', 'P', 'H', 'G', 'T', 'L', '*'], dtype='<U1'), 'Seq2_long_ALL6_frames': array(['M', 'V', 'M', 'A', 'I', '-', '-', '-', 'Y', 'L', '-', '-', '-',
       '-', '-', '-', '-', '-', '-', '-', '-', 'S', 'I', 'P', 'W', 'F',
       'K', 'H', 'L', 'S', '-', '-', '-', '-', 'A', 'W', '-', 'S', '-',
       '-', '-', 'K', 'H', '-', '-', 'P', '*'], dtype='<U1'), 'Seq3_long_ALL6_frames': array(['M', '-', 'K', 'L', 'R', 'P', 'W', '-', '-', 'L', 'R', '-', '-',
       '-', '-', '-', '-', 'M', 'T', 'L', 'D', 'L', 'C', '-', '-', '-',
       '-', 'V', 'K', 'D', 'D', 'H', 'D', 'L', 'D', 'L', 'C', 'G', 'K',
       'D', 'E', 'P', '-', '-', '-', '-', '*'], dtype='<U1')}


In [66]:
def back_translate_alignment(amino_alignment_dict, df):
    """
    Convert aligned AA sequences back to codon-preserving nucleotide alignments.
    """

    codon_alignment = {}
    
    # Convert all ORFs into lists of STR codons
    nuc_map = {}
    for _, row in df.iterrows():
        seq_id = row["id"]
        orf = str(row["orf"])     # <-- convert Seq to string safely
        codons = [str(orf[i:i+3]) for i in range(0, len(orf), 3)]
        nuc_map[seq_id] = codons

    # Build back-translated alignment
    for seq_id, aa_alignment in amino_alignment_dict.items():

        codons = nuc_map[seq_id]
        codon_idx = 0
        codon_aligned = []

        for aa in aa_alignment:

            if aa == "-":
                codon_aligned.append("---")
            else:
                codon_aligned.append(codons[codon_idx])
                codon_idx += 1

        codon_alignment[seq_id] = "".join(codon_aligned)

    return codon_alignment


In [67]:
codon_alignment = back_translate_alignment(alignment, df1)
codon_alignment

{'Seq1_long_ALL6_frames': 'ATGGTTAACGCTATCCCATGGTTCAAACTACGGTCATCTAAACATGGTTCAGTTACG---GTTTCATGCCCTAACTCATCGGGCATCAGTTACCCTATCAGGCCATGGTCAAGCTACATTCAGCCTCATGGTACCCTTTAG',
 'Seq2_long_ALL6_frames': 'ATGGTCATGGCCATC---------TACTTA---------------------------------AGCATCCCATGGTTCAAACATCTATCC------------GCATGG---TCT---------AAGCAT------CCTTAA',
 'Seq3_long_ALL6_frames': 'ATG---AAACTTAGACCATGG------CTTAGA------------------ATGACCTTAGACCTATGC------------GTTAAGGATGACCATGACTTAGACCTATGCGGTAAGGATGAACCT------------TAG'}